# 回帰分析：検定（1）

前回は、都道府県別の最低賃金と物価水準指数の関係を単回帰モデルで推定した。  
今回は、推定された回帰係数が偶然のばらつきによって得られたものなのか、それともデータから統計的に意味のある関係が読み取れるのかを考える。

中心となる道具は、標準誤差、t検定、p値、信頼区間である。


## 1. 今回の目標

回帰分析における検定では、推定値そのものだけでなく、推定値の不確実性を扱う。

推定値は「標本から得られた1つの値」である。検定は、その値がどの程度確かなものなのかを調べる作業である。


## 2. ライブラリの読み込み


In [ ]:
%pip install japanize-matplotlib-jlite -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
import japanize_matplotlib_jlite


## 3. 使用するデータ

前回と同じく、都道府県別の最低賃金と物価水準指数を用いる。最低賃金は令和7年度の地域別最低賃金、物価水準指数は2024年の消費者物価地域差指数である。


In [ ]:
# 都道府県別データ
# 出典1：厚生労働省「令和7年度 地域別最低賃金 全国一覧」
# URL：https://www.mhlw.go.jp/content/11200000/001571192.pdf
#
# 出典2：総務省統計局「消費者物価地域差指数－小売物価統計調査（構造編）2024年（令和6年）結果－」別表1
# URL：https://www.stat.go.jp/data/kouri/kouzou/pdf/g_2024.pdf
# 消費者物価地域差指数は、地域間の物価水準の差を表す指数

prefectures = [
    "北海道", "青森", "岩手", "宮城", "秋田", "山形", "福島",
    "茨城", "栃木", "群馬", "埼玉", "千葉", "東京", "神奈川",
    "新潟", "富山", "石川", "福井", "山梨", "長野", "岐阜",
    "静岡", "愛知", "三重", "滋賀", "京都", "大阪", "兵庫",
    "奈良", "和歌山", "鳥取", "島根", "岡山", "広島", "山口",
    "徳島", "香川", "愛媛", "高知", "福岡", "佐賀", "長崎",
    "熊本", "大分", "宮崎", "鹿児島", "沖縄"
]

minimum_wages = [
    1075, 1029, 1031, 1038, 1031, 1032, 1033,
    1074, 1068, 1063, 1141, 1140, 1226, 1225,
    1050, 1062, 1054, 1053, 1052, 1061, 1065,
    1097, 1140, 1087, 1080, 1122, 1177, 1116,
    1051, 1045, 1030, 1033, 1047, 1085, 1043,
    1046, 1036, 1033, 1023, 1057, 1030, 1031,
    1034, 1035, 1023, 1026, 1023
]

price_level_index = [
    101.9, 98.5, 100.0, 100.6, 99.2, 101.4, 98.8,
    97.5, 97.6, 96.2, 100.3, 101.2, 104.0, 103.3,
    98.0, 98.6, 99.5, 99.3, 97.7, 97.9, 97.1,
    98.3, 98.1, 98.7, 98.6, 101.1, 99.3, 99.2,
    98.1, 98.2, 98.9, 100.0, 97.7, 98.7, 99.9,
    99.3, 98.6, 98.6, 100.0, 98.0, 97.7, 99.3,
    99.4, 97.4, 97.0, 96.4, 100.2
]


In [ ]:
df = pd.DataFrame({
    "prefecture": prefectures,
    "minimum_wage": minimum_wages,
    "price_index": price_level_index,
})

df.head()


In [ ]:
# 行数と列数を確認する
print(df.shape)


## 4. 前回の回帰モデルを再推定する

被説明変数を最低賃金、説明変数を物価水準指数とする単回帰モデルを考える。

$
Y_i = \beta_0 + \beta_1 X_i + u_i
$

ここで、$Y_i$ は都道府県 $i$ の最低賃金、$X_i$ は物価水準指数、$u_i$ は誤差項である。


In [ ]:
# 被説明変数
Y = df["minimum_wage"]

# 説明変数
X = df[["price_index"]]

# 切片に対応する定数項を追加する
X = sm.add_constant(X)

model = sm.OLS(Y, X)
result = model.fit()

result.params


In [ ]:
coef_table = pd.DataFrame({
    "推定値": result.params,
})
coef_table


推定された回帰式は、次の形で表される。

$
\widehat{Y}_i = \widehat{\beta}_0 + \widehat{\beta}_1 X_i
$

傾き $\widehat{\beta}_1$ は、物価水準指数が1ポイント高い都道府県では、最低賃金が平均的に何円高いと推定されるかを表す。


In [ ]:
x_line = np.linspace(df["price_index"].min(), df["price_index"].max(), 100)
y_line = result.params["const"] + result.params["price_index"] * x_line

plt.figure(figsize=(8, 6))
plt.scatter(df["price_index"], df["minimum_wage"], alpha=0.7, label="実際のデータ")
plt.plot(x_line, y_line, label="推定された回帰直線")

plt.title("物価水準と最低賃金、推定された回帰直線")
plt.xlabel("消費者物価地域差指数、全国平均＝100 (2024年)")
plt.ylabel("最低賃金 (令和7年度)")
plt.legend()
plt.show()


## 5. 推定値にはばらつきがある

推定値は、手元の標本から計算された値である。別の標本を観測して同じ分析を行えば、推定値は少し異なる可能性がある。

この「標本が変われば推定値も変わる」という考え方が、検定と信頼区間の出発点である。

「推定された傾きは0ではないように見えるが、これは偶然のばらつきだけでも起こりうる大きさなのか？」

## 6. 残差平方和と誤差分散の推定

まず、推定された回帰式から当てはめ値と残差を求める。

$
\widehat{u}_i = Y_i - \widehat{Y}_i
$

残差平方和は、残差の二乗を合計したものである。

$
RSS = \sum_{i=1}^{n} \widehat{u}_i^2
$

単回帰モデルでは、切片と傾きの2つの係数を推定しているため、誤差分散は次の式で推定する。

$
\widehat{\sigma}^2 = \frac{RSS}{n-2}
$

In [ ]:
df["fitted"] = result.fittedvalues
df["residual"] = result.resid

n = len(df)
k = 2
rss = np.sum(df["residual"] ** 2)
sigma2_hat = rss / (n - k)
sigma_hat = np.sqrt(sigma2_hat)

print("標本サイズ n:", n)
print("推定した係数の数 k:", k)
print("自由度 n-k:", n - k)
print("残差平方和 RSS:", rss)
print("誤差分散の推定値:", sigma2_hat)
print("誤差の標準偏差の推定値:", sigma_hat)


残差平方和が大きいほど、実際のデータ点は回帰直線から大きく離れている。したがって、推定された係数にも大きな不確実性が残る。

## 7. 回帰係数の標準誤差

単回帰モデルでは、傾きの推定値の標準誤差は次の式で求められる。

$
SE(\widehat{\beta}_1)
= \sqrt{\frac{\widehat{\sigma}^2}{\sum_{i=1}^{n}(X_i - \bar{X})^2}}
$

切片の推定値の標準誤差は次の式で求められる。

$
SE(\widehat{\beta}_0)
= \sqrt{\widehat{\sigma}^2 \left( \frac{1}{n} + \frac{\bar{X}^2}{\sum_{i=1}^{n}(X_i - \bar{X})^2} \right)}
$

標準誤差は、推定値がどの程度ばらつきうるかを表す尺度である。


In [ ]:
x = df["price_index"].to_numpy()
x_bar = np.mean(x)
Sxx = np.sum((x - x_bar) ** 2)

se_beta1 = np.sqrt(sigma2_hat / Sxx)
se_beta0 = np.sqrt(sigma2_hat * (1 / n + x_bar ** 2 / Sxx))

manual_se_table = pd.DataFrame({
    "推定値": [result.params["const"], result.params["price_index"]],
    "標準誤差（手計算）": [se_beta0, se_beta1],
}, index=["const", "price_index"])

manual_se_table


In [ ]:
se_compare = pd.DataFrame({
    "手計算": [se_beta0, se_beta1],
    "statsmodels": result.bse,
}, index=["const", "price_index"])

se_compare


手計算で求めた標準誤差は、statsmodels の結果と一致する。

傾きの推定値が大きくても、標準誤差も大きければ、その推定値は不確実である。反対に、標準誤差が小さければ、推定値は比較的精密である。


## 8. 仮説検定の考え方

仮説検定では、まず帰無仮説と対立仮説を設定する。

傾きの係数について検定する場合、典型的には次の仮説を考える。

$
H_0: \beta_1 = 0
$

$
H_1: \beta_1 \neq 0
$

帰無仮説 $H_0$ は「物価水準指数と最低賃金の間に線形の関係はない」という仮説である。対立仮説 $H_1$ は「物価水準指数と最低賃金の間に線形の関係がある」という仮説である。


## 9. 傾きに関する $t$ 検定

$t$ 検定では、推定値が帰無仮説の値から何個分の標準誤差だけ離れているかを調べる。

傾きの係数について、帰無仮説を $\beta_1=0$ とすると、t値は次の式で計算される。

$
t
= \frac{\widehat{\beta}_1 - 0}{SE(\widehat{\beta}_1)}
$

$t$ 値の絶対値が大きいほど、推定値は帰無仮説の値から遠い。


In [ ]:
beta1_null = 0
beta1_hat = result.params["price_index"]
se_beta1_sm = result.bse["price_index"]
df_resid = int(result.df_resid)

t_beta1 = (beta1_hat - beta1_null) / se_beta1_sm
p_beta1 = 2 * stats.t.sf(abs(t_beta1), df=df_resid)

print("傾きの推定値:", beta1_hat)
print("傾きの標準誤差:", se_beta1_sm)
print("t値:", t_beta1)
print("自由度:", df_resid)
print("両側検定のp値:", p_beta1)


In [ ]:
alpha = 0.05

if p_beta1 < alpha:
    print("5%有意水準で帰無仮説を棄却する")
else:
    print("5%有意水準で帰無仮説を棄却しない")


この検定では、p値が有意水準5%より小さい場合、$\beta_1=0$ という帰無仮説を棄却する。

このとき、「物価水準指数と最低賃金の間には、統計的に有意な線形関係がある」と表現する。


## 10. p値の意味

p値は、帰無仮説が正しいと仮定したときに、今回のデータで得られた検定統計量と同じか、それより極端な値が得られる確率である。

ここでは、帰無仮説 \(\beta_1=0\) が正しいと仮定する。そのもとで、今回のt値のように絶対値が大きい値が出る確率を計算する。これがp値である。

p値が小さいほど、帰無仮説のもとでは今回のデータが起こりにくいことを意味する。


In [ ]:
t_grid = np.linspace(-5, 5, 500)
density = stats.t.pdf(t_grid, df=df_resid)
abs_t = abs(t_beta1)

plt.figure(figsize=(8, 5))
plt.plot(t_grid, density)
plt.fill_between(t_grid[t_grid <= -abs_t], density[t_grid <= -abs_t], alpha=0.3)
plt.fill_between(t_grid[t_grid >= abs_t], density[t_grid >= abs_t], alpha=0.3)
plt.axvline(-abs_t, linestyle="--")
plt.axvline(abs_t, linestyle="--")

plt.title("t分布と両側検定のp値")
plt.xlabel("t値")
plt.ylabel("密度")
plt.show()


p値は「帰無仮説が正しい確率」ではない。また、p値が小さいことは「効果が大きい」ことを直接意味しない。

p値は、帰無仮説のもとで観測された結果がどの程度起こりにくいかを表す値である。


## 11. statsmodels で検定結果を確認する

statsmodels は、推定値、標準誤差、t値、p値をまとめて計算している。


In [ ]:
test_table = pd.DataFrame({
    "推定値": result.params,
    "標準誤差": result.bse,
    "t値": result.tvalues,
    "p値": result.pvalues,
})

test_table


`price_index` の行が、物価水準指数の傾きに関する検定結果である。

`const` は切片を表す。切片の検定結果も表示されるが、切片の経済的解釈には注意が必要である。


## 12. 信頼区間

信頼区間は、推定値の不確実性を区間として表したものである。  

ただし、ここでは、母数 $\beta_1$ は固定された未知の値である。確率的に変動するのは、標本、推定値、そして標本から計算される信頼区間である。

95%信頼区間とは、同じ方法で標本抽出と区間計算を何度も繰り返したとき、作られる区間の約95%が真の母数を含むような手続きによって得られる区間である。したがって、今回計算された特定の区間について、「$\beta_1$ が95%の確率でこの区間に入っている」とは解釈しない。

この点は、ベイズ統計における信用区間と異なる。ベイズの信用区間は、事前分布とデータから得られる事後分布に基づき、母数がその区間に入る事後確率として解釈される。一方、この講義で扱う信頼区間は、頻度主義の回帰分析における信頼区間である。

日常的な説明としては、「データとモデルの仮定のもとで、$\beta_1$ の値として矛盾しにくい範囲」と捉えると理解しやすい。ただし、厳密には、確率がかかっているのは母数そのものではなく、区間を作る手続きである。

95%信頼区間は、次の式で求められる。

$
\widehat{\beta}_1
\pm t_{0.975, n-2} \times SE(\widehat{\beta}_1)
$

ここで、$t_{0.975, n-2}$ は自由度 $n-2$ のt分布における97.5%点である。両側5%の信頼区間であるため、片側に2.5%ずつを置く。


In [ ]:
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha / 2, df=df_resid)

ci_beta1_lower = beta1_hat - t_crit * se_beta1_sm
ci_beta1_upper = beta1_hat + t_crit * se_beta1_sm

print("傾きの95%信頼区間:", (ci_beta1_lower, ci_beta1_upper))


In [ ]:
conf_int = result.conf_int(alpha=0.05)
conf_int.columns = ["95%信頼区間 下限", "95%信頼区間 上限"]
conf_int


傾きの95%信頼区間が0を含まない場合、5%有意水準の両側t検定では \(\beta_1=0\) を棄却する。

信頼区間は、検定結果だけでなく、効果の大きさの範囲を確認するためにも重要である。


## 13. 切片の検定

statsmodels の表には、切片 `const` のt値とp値も表示される。

切片は \(X=0\) のときの \(Y\) の予測値である。しかし、このデータでは物価水準指数が0の都道府県は存在しない。したがって、切片の検定結果をそのまま経済的に重視する必要は小さい。


In [ ]:
beta0_hat = result.params["const"]
se_beta0_sm = result.bse["const"]
t_beta0 = result.tvalues["const"]
p_beta0 = result.pvalues["const"]

print("切片の推定値:", beta0_hat)
print("切片の標準誤差:", se_beta0_sm)
print("切片のt値:", t_beta0)
print("切片のp値:", p_beta0)


検定では、統計的に有意かどうかだけでなく、その係数を解釈する意味があるかどうかも確認する必要がある。

切片の値は回帰直線を描くうえで必要であるが、この例では政策的・経済的な解釈の中心ではない。


## 14. 統計的有意性と効果の大きさ

統計的に有意であることは、係数が実質的に大きいことを意味しない。

この例では、傾きの推定値は「物価水準指数が1ポイント高いと、最低賃金が平均的に何円高いか」を表す。したがって、p値だけでなく、係数の大きさそのものを確認する必要がある。


In [ ]:
for delta in [1, 3, 5]:
    change = result.params["price_index"] * delta
    print(f"物価水準指数が{delta}ポイント高い場合の予測値の差: {change:.2f}円")


p値は「関係があるかどうか」を判断するための情報である。一方、係数の大きさは「どの程度の関係か」を判断するための情報である。

回帰分析の解釈では、この2つを分けて考える必要がある。


## 15. 検定結果の書き方

検定結果は、推定値、標準誤差、t値、p値、信頼区間を組み合わせて書くとよい。

このデータの例では、次のようにまとめられる。

「物価水準指数の係数は正であり、5%有意水準で統計的に有意である。したがって、物価水準指数が高い都道府県ほど、最低賃金が高い傾向があるといえる。ただし、この結果は相関関係を示すものであり、物価水準が最低賃金を直接引き上げる因果効果を示すものではない。」


In [ ]:
summary_table = pd.DataFrame({
    "推定値": result.params,
    "標準誤差": result.bse,
    "t値": result.tvalues,
    "p値": result.pvalues,
})
summary_table["95%信頼区間 下限"] = conf_int["95%信頼区間 下限"]
summary_table["95%信頼区間 上限"] = conf_int["95%信頼区間 上限"]

summary_table.round(4)
